In [123]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_PATH = Path("../data/raw")

print("Data quality audit started.")

Data quality audit started.


In [124]:
users = pd.read_csv(
    RAW_DATA_PATH / "users.csv",
    parse_dates=["signup_date"]
)

creators = pd.read_csv(
    RAW_DATA_PATH / "creators.csv",
    parse_dates=["signup_date"]
)

content = pd.read_csv(
    RAW_DATA_PATH / "content.csv",
    parse_dates=["created_at"]
)

interactions = pd.read_csv(
    RAW_DATA_PATH / "interactions.csv",
    parse_dates=["timestamp", "content_created_at"]
)

print("Users:", users.shape)
print("Creators:", creators.shape)
print("Content:", content.shape)
print("Interactions:", interactions.shape)

Users: (5000, 7)
Creators: (500, 5)
Content: (10000, 9)
Interactions: (250000, 22)


In [125]:
datasets = {
    "users": users,
    "creators": creators,
    "content": content,
    "interactions": interactions
}

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"{name.upper()}")
    print(f"{'=' * 50}")
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isna().sum().sum())


USERS
Shape: (5000, 7)
Duplicate rows: 0
Missing values: 0

CREATORS
Shape: (500, 5)
Duplicate rows: 0
Missing values: 0

CONTENT
Shape: (10000, 9)
Duplicate rows: 0
Missing values: 0

INTERACTIONS
Shape: (250000, 22)
Duplicate rows: 0
Missing values: 0


In [126]:
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    
    missing = (
        df.isna()
        .sum()
        .to_frame("missing_count")
    )
    
    missing["missing_pct"] = (
        missing["missing_count"] / len(df) * 100
    ).round(2)
    
    print(
        missing[missing["missing_count"] > 0]
        .sort_values("missing_pct", ascending=False)
    )


--- users ---
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []

--- creators ---
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []

--- content ---
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []

--- interactions ---
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []


In [127]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    
    print(
        f"{name}: "
        f"{duplicate_count:,} duplicate rows"
    )

users: 0 duplicate rows
creators: 0 duplicate rows
content: 0 duplicate rows
interactions: 0 duplicate rows


In [128]:
primary_keys = {
    "users": "user_id",
    "creators": "creator_id",
    "content": "content_id"
}

for name, key in primary_keys.items():
    df = datasets[name]
    
    duplicate_keys = df[key].duplicated().sum()
    
    print(
        f"{name}.{key}: "
        f"{duplicate_keys:,} duplicate IDs"
    )

users.user_id: 0 duplicate IDs
creators.creator_id: 0 duplicate IDs
content.content_id: 0 duplicate IDs


In [129]:
invalid_content_creators = (
    ~content["creator_id"].isin(creators["creator_id"])
)

print(
    "Content rows with invalid creator IDs:",
    invalid_content_creators.sum()
)

Content rows with invalid creator IDs: 0


In [130]:
invalid_interaction_users = (
    ~interactions["user_id"].isin(users["user_id"])
)

invalid_interaction_content = (
    ~interactions["content_id"].isin(content["content_id"])
)

invalid_interaction_creators = (
    ~interactions["creator_id"].isin(creators["creator_id"])
)

print(
    "Invalid interaction user IDs:",
    invalid_interaction_users.sum()
)

print(
    "Invalid interaction content IDs:",
    invalid_interaction_content.sum()
)

print(
    "Invalid interaction creator IDs:",
    invalid_interaction_creators.sum()
)

Invalid interaction user IDs: 0
Invalid interaction content IDs: 0
Invalid interaction creator IDs: 0


In [131]:
print("Users")
print("Following count < 0:",
      (users["following_count"] < 0).sum())

print("\nCreators")
print("Followers < 0:",
      (creators["followers"] < 0).sum())

print("\nContent")
print("Duration <= 0:",
      (content["duration"] <= 0).sum())

print("\nInteractions")
print("Watch time < 0:",
      (interactions["watch_time"] < 0).sum())

print(
    "Completion rate outside [0, 1]:",
    (
        (interactions["completion_rate"] < 0) |
        (interactions["completion_rate"] > 1)
    ).sum()
)

Users
Following count < 0: 0

Creators
Followers < 0: 0

Content
Duration <= 0: 0

Interactions
Watch time < 0: 0
Completion rate outside [0, 1]: 0


In [132]:
print(
    "Likes without click:",
    (
        (interactions["liked"] == 1) &
        (interactions["clicked"] == 0)
    ).sum()
)

print(
    "Saves without click:",
    (
        (interactions["saved"] == 1) &
        (interactions["clicked"] == 0)
    ).sum()
)

print(
    "Shares without click:",
    (
        (interactions["shared"] == 1) &
        (interactions["clicked"] == 0)
    ).sum()
)

print(
    "Comments without click:",
    (
        (interactions["commented"] == 1) &
        (interactions["clicked"] == 0)
    ).sum()
)

print(
    "Recreations without click:",
    (
        (interactions["recreated"] == 1) &
        (interactions["clicked"] == 0)
    ).sum()
)

Likes without click: 0
Saves without click: 0
Shares without click: 0
Comments without click: 0
Recreations without click: 0


In [133]:
print(
    "Watch time without click:",
    (
        (interactions["clicked"] == 0) &
        (interactions["watch_time"] > 0)
    ).sum()
)

Watch time without click: 0


In [134]:
print(
    "Watch time greater than duration:",
    (
        interactions["watch_time"] >
        interactions["duration"]
    ).sum()
)

Watch time greater than duration: 0


In [135]:
print("Interaction timestamp range:")
print("Minimum:", interactions["timestamp"].min())
print("Maximum:", interactions["timestamp"].max())

print("\nContent creation timestamp range:")
print("Minimum:", content["created_at"].min())
print("Maximum:", content["created_at"].max())

Interaction timestamp range:
Minimum: 2025-06-01 23:55:16
Maximum: 2026-08-30 23:59:46

Content creation timestamp range:
Minimum: 2025-06-01 03:58:39
Maximum: 2026-08-30 23:55:53


In [136]:
interaction_before_creation = (
    interactions["timestamp"] <
    interactions["content_created_at"]
)

print(
    "Interactions before content creation:",
    interaction_before_creation.sum()
)

Interactions before content creation: 0


In [137]:
print("User countries:")
print(users["country"].value_counts())

print("\nUser age groups:")
print(users["age_group"].value_counts())

print("\nCreator types:")
print(creators["creator_type"].value_counts())

print("\nContent types:")
print(content["content_type"].value_counts())

print("\nContent genres:")
print(content["genre"].value_counts())

User countries:
country
Germany           757
Canada            744
India             722
United Kingdom    722
United States     711
Singapore         703
Australia         641
Name: count, dtype: int64

User age groups:
age_group
18-24    1785
25-34    1520
35-44     973
45+       464
13-17     258
Name: count, dtype: int64

Creator types:
creator_type
AI Creator           112
Artist               111
Influencer            98
Community Creator     90
Filmmaker             89
Name: count, dtype: int64

Content types:
content_type
mini     4524
image    2023
cine     1941
video    1512
Name: count, dtype: int64

Content genres:
genre
Horror         1040
Romance        1024
Action         1014
Drama          1013
Documentary    1012
Mystery         996
Comedy          993
Animation       990
Fantasy         970
Sci-Fi          948
Name: count, dtype: int64


In [138]:
def iqr_outlier_summary(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    outliers = (
        (df[column] < lower) |
        (df[column] > upper)
    )
    
    return {
        "column": column,
        "Q1": q1,
        "Q3": q3,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(outliers.sum()),
        "outlier_pct": round(outliers.mean() * 100, 2)
    }

In [139]:
outlier_checks = []

for column in ["followers"]:
    outlier_checks.append(
        iqr_outlier_summary(creators, column)
    )

for column in ["duration"]:
    outlier_checks.append(
        iqr_outlier_summary(content, column)
    )

for column in ["watch_time"]:
    outlier_checks.append(
        iqr_outlier_summary(interactions, column)
    )

pd.DataFrame(outlier_checks)

,column,Q1,Q3,lower_bound,upper_bound,outlier_count,outlier_pct
0,followers,20.0,109.00,-113.500,242.500,49,9.80
1,duration,12.6,58.40,-56.100,127.100,597,5.97
2,watch_time,0.0,10.21,-15.315,25.525,35078,14.03


In [140]:
audit_summary = []

for name, df in datasets.items():
    audit_summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum()
    })

audit_summary = pd.DataFrame(audit_summary)

audit_summary

,dataset,rows,columns,duplicate_rows,missing_cells
0,users,5000,7,0,0
1,creators,500,5,0,0
2,content,10000,9,0,0
3,interactions,250000,22,0,0


In [141]:
import sys
import importlib

# Make sure project root is first
sys.path.insert(0, "../")

# Remove the old cached module
for module_name in list(sys.modules):
    if module_name == "src.data.cleaning":
        del sys.modules[module_name]

# Import the freshly saved file
import src.data.cleaning as cleaning

print("Loaded from:", cleaning.__file__)
print("clean_users exists:", hasattr(cleaning, "clean_users"))
print("Available functions:", [
    x for x in dir(cleaning)
    if x.startswith("clean_") or x.startswith("validate_")
])


Loaded from: c:\Users\ADITYA\Documents\ds-projects\aicines-content-intelligence\notebooks\..\src\data\cleaning.py
clean_users exists: True
Available functions: ['clean_content', 'clean_creators', 'clean_interactions', 'clean_users', 'validate_interactions']


In [142]:
from src.data.cleaning import (
    clean_users,
    clean_creators,
    clean_content,
    clean_interactions,
    validate_interactions
)

print("Import successful!")


Import successful!


In [143]:
from pathlib import Path
import src.data.cleaning as cleaning

path = Path(cleaning.__file__).resolve()

print(path)
print("Size:", path.stat().st_size)
print("Contains clean_users:", "def clean_users" in path.read_text(encoding="utf-8"))


C:\Users\ADITYA\Documents\ds-projects\aicines-content-intelligence\src\data\cleaning.py
Size: 6666
Contains clean_users: True


In [144]:
import sys

sys.path.append("../")

from src.data.cleaning import (
    clean_users,
    clean_creators,
    clean_content,
    clean_interactions,
    validate_interactions
)

In [145]:
users_clean = clean_users(users)

creators_clean = clean_creators(creators)

content_clean = clean_content(content)

interactions_clean = clean_interactions(
    interactions,
    users_clean,
    creators_clean,
    content_clean
)

print("Users:", users_clean.shape)
print("Creators:", creators_clean.shape)
print("Content:", content_clean.shape)
print("Interactions:", interactions_clean.shape)

Users: (5000, 7)
Creators: (500, 5)
Content: (10000, 9)
Interactions: (250000, 22)


In [146]:
validation_results = validate_interactions(
    interactions_clean
)

validation_results

{'duplicate_rows': 0,
 'negative_watch_time': 0,
 'completion_rate_out_of_range': 0,
 'watch_time_exceeds_duration': 0}

In [147]:
PROCESSED_DATA_PATH = Path("../data/processed")

PROCESSED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

users_clean.to_csv(
    PROCESSED_DATA_PATH / "users_clean.csv",
    index=False
)

creators_clean.to_csv(
    PROCESSED_DATA_PATH / "creators_clean.csv",
    index=False
)

content_clean.to_csv(
    PROCESSED_DATA_PATH / "content_clean.csv",
    index=False
)

interactions_clean.to_csv(
    PROCESSED_DATA_PATH / "interactions_clean.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [148]:
for name, df in {
    "users_clean": users_clean,
    "creators_clean": creators_clean,
    "content_clean": content_clean,
    "interactions_clean": interactions_clean
}.items():

    print(f"\n{name}")
    print("-" * 40)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Missing cells:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())


users_clean
----------------------------------------
Rows: 5000
Columns: 7
Missing cells: 0
Duplicate rows: 0

creators_clean
----------------------------------------
Rows: 500
Columns: 5
Missing cells: 0
Duplicate rows: 0

content_clean
----------------------------------------
Rows: 10000
Columns: 9
Missing cells: 0
Duplicate rows: 0

interactions_clean
----------------------------------------
Rows: 250000
Columns: 22
Missing cells: 0
Duplicate rows: 0
